# Task 1: Exploratory Data Analysis (EDA)

## Objective
Develop a thorough understanding of the financial news dataset through exploratory analysis. This involves descriptive statistics, text analysis, and time-series analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords

# Set visual style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Download stopwords if not already present
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

## 1. Load Data

In [ ]:
data_path = "../data/raw/raw_analyst_ratings.csv"

if os.path.exists(data_path):
    # Note: Loading only a subset if the file is too large for memory, 
    # but for full analysis we'll use the whole thing.
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    print("Error: Dataset not found at specified path.")

## 2. Preliminary Data Inspection

In [ ]:
df.head()

In [ ]:
df.info()

## 3. Descriptive Statistics

### Headline Length Distribution

In [ ]:
df['headline_len'] = df['headline'].apply(len)
sns.histplot(df['headline_len'], bins=50, kde=True)
plt.title('Distribution of Headline Lengths')
plt.xlabel('Character Count')
plt.ylabel('Frequency')
plt.show()

print(df['headline_len'].describe())

### Articles per Publisher

In [ ]:
publisher_counts = df['publisher'].value_counts()
top_publishers = publisher_counts.head(10)

sns.barplot(x=top_publishers.values, y=top_publishers.index, palette='viridis')
plt.title('Top 10 Most Active Publishers')
plt.xlabel('Number of Articles')
plt.ylabel('Publisher')
plt.show()

## 4. Time Series Analysis of News Volume

In [ ]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Check for null dates and drop if necessary for time analysis
df_time = df.dropna(subset=['date'])

# Plot resampled by day
df_time.set_index('date').resample('D').size().plot(color='teal')
plt.title('Daily News Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Articles')
plt.show()

In [ ]:
# Analyze publishing times (Hour of the day)
df_time['hour'] = df_time['date'].dt.hour
sns.countplot(x='hour', data=df_time, palette='magma')
plt.title('Article Publication Distribution by Hour of Day')
plt.xlabel('Hour (24h format)')
plt.ylabel('Article Count')
plt.show()

## 5. Text Analysis: Topic & Keyword Extraction

In [ ]:
def get_top_n_words(corpus, n=None, n_gram_range=(1,1)):
    """
    List the top n words/phrases in a vocabulary according to occurrence in a text corpus.
    """
    vec = CountVectorizer(stop_words='english', ngram_range=n_gram_range).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

# Sample headlines for faster processing if dataset is huge
sample_headlines = df['headline'].dropna().sample(min(50000, len(df)))

# Get top Unigrams (Single words)
common_words = get_top_n_words(sample_headlines, 20)
df_words = pd.DataFrame(common_words, columns=['word', 'count'])

sns.barplot(x='count', y='word', data=df_words, palette='Blues_r')
plt.title('Top 20 Most Common Keywords in Headlines')
plt.show()

In [ ]:
# Get top Bigrams (Two-word phrases like "FDA approval", "Price target")
common_bigrams = get_top_n_words(sample_headlines, 20, n_gram_range=(2,2))
df_bigrams = pd.DataFrame(common_bigrams, columns=['phrase', 'count'])

sns.barplot(x='count', y='phrase', data=df_bigrams, palette='Reds_r')
plt.title('Top 20 Most Common Bigrams (Phrases)')
plt.show()

## 6. Summary of Findings
1. **Headline Length:** Insights about character count distribution.
2. **Publishers:** Identification of top contributors.
3. **Time Trends:** Spikes in publication volume and daily publication patterns.
4. **Topics:** Recurring themes identified through keyword and phrase analysis.